# Amazon ML Challenge 2026: Stage 4 — Gated Cross-Encoder Ambiguity Refinement
### Resolving High-Entropy Boundary Pairs ($0.45 \le P \le 0.70$) with MiniLM Cross-Encoder

**Purpose:**
Bi-encoders (FAISS) have known syntactic limitations for nuanced textual differences (e.g. "Starbucks Coffee" vs "Starbucks Cafe", or subtle house number shifts). LightGBM classifies ~90-95% of pairs with high confidence, but leaves ~5% in an ambiguous band.

This notebook loads the scored pairs from Stage 6 v2, passes ONLY the ambiguous band through `cross-encoder/ms-marco-MiniLM-L-6-v2`, and blends the probabilities:
$$P_{\text{final}} = 0.5 \cdot P_{\text{GBDT}} + 0.5 \cdot P_{\text{CrossEncoder}}$$

**Expected Impact:** Pushes Macro $F_{0.5}$ from ~95-96% into the **98.7%+ leaderboard tier** by eliminating false positives and recovering borderline true positives.

In [ ]:
!pip install -q transformers torch pyarrow

import os, sys, gc, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

IS_KAGGLE = os.path.exists('/kaggle')

def find_file(pattern):
    search_root = Path('/kaggle/input') if IS_KAGGLE else Path('.')
    matches = list(search_root.rglob(pattern))
    if not matches:
        return None
    matches.sort(key=lambda p: (p.stat().st_size, p.stat().st_mtime), reverse=True)
    return matches[0]

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load Candidate Pairs and Ambiguous Band
WORK_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('work')
OUTPUT_DIR = WORK_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
BATCH_SIZE = 256 if DEVICE == 'cuda' else 32
MAX_LEN = 128
BLEND_ALPHA = 0.5

print(f"Loading Cross-Encoder model: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

print("Cross-Encoder ready for inference!")